In [ ]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("03_Bronze_to_Silver")
    .master("local[2]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .config("spark.ui.enabled", "false")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")
print("Spark OK :", spark.version)


In [ ]:
BRONZE = "C:/Users/smagu/retail-data-platform/data/bronze"
SILVER = "C:/Users/smagu/retail-data-platform/data/silver"

clients_pd      = pd.read_csv(BRONZE + "/clients.csv")
employes_pd     = pd.read_csv(BRONZE + "/employes.csv")
fournisseurs_pd = pd.read_csv(BRONZE + "/fournisseurs.csv")
produits_pd     = pd.read_csv(BRONZE + "/produits.csv")
ventes_pd       = pd.read_csv(BRONZE + "/ventes.csv")

print("Tables Bronze chargees :")
for nom, df in [("clients", clients_pd), ("employes", employes_pd),
                ("fournisseurs", fournisseurs_pd), ("produits", produits_pd),
                ("ventes", ventes_pd)]:
    print("  " + nom + " : " + str(len(df)) + " lignes")


In [ ]:
# Nettoyage : types et colonnes derivees
clients_pd["ClientID"]           = clients_pd["ClientID"].astype(int)
employes_pd["EmployeID"]         = employes_pd["EmployeID"].astype(int)
fournisseurs_pd["FournisseurID"] = fournisseurs_pd["FournisseurID"].astype(int)
produits_pd["PrixUnitaire"]      = produits_pd["PrixUnitaire"].astype(float)

ventes_pd["DateVente"]      = pd.to_datetime(ventes_pd["DateVente"])
ventes_pd["Annee"]          = ventes_pd["DateVente"].dt.year
ventes_pd["Mois"]           = ventes_pd["DateVente"].dt.month
ventes_pd["MontantTotal"]   = ventes_pd["MontantTotal"].astype(float)
ventes_pd["QuantiteVendue"] = ventes_pd["QuantiteVendue"].astype(int)

print("Nettoyage de base OK")
print(ventes_pd.dtypes)


In [ ]:
# Filtrage des valeurs aberrantes — methode IQR sur QuantiteVendue
Q1  = ventes_pd["QuantiteVendue"].quantile(0.25)
Q3  = ventes_pd["QuantiteVendue"].quantile(0.75)
IQR = Q3 - Q1
seuil_haut = Q3 + 1.5 * IQR

outliers     = ventes_pd[ventes_pd["QuantiteVendue"] > seuil_haut]
ventes_clean = ventes_pd[ventes_pd["QuantiteVendue"] <= seuil_haut].copy()

print("=== DETECTION OUTLIERS (IQR sur QuantiteVendue) ===")
print("Q1=" + str(int(Q1)) + "  |  Q3=" + str(int(Q3)) + "  |  IQR=" + str(int(IQR)) + "  |  Seuil haut=" + str(int(seuil_haut)))
print("Lignes avant  : " + str(len(ventes_pd)))
print("Outliers      : " + str(len(outliers)))
print("Lignes apres  : " + str(len(ventes_clean)))
print()
print("Lignes supprimees :")
print(outliers[["VenteID", "ProduitID", "QuantiteVendue", "MontantTotal"]].to_string(index=False))


In [ ]:
print("=== STATS MontantTotal AVANT filtrage ===")
print(ventes_pd["MontantTotal"].describe().apply(lambda x: "{:,.0f}".format(x)))
print()
print("=== STATS MontantTotal APRES filtrage ===")
print(ventes_clean["MontantTotal"].describe().apply(lambda x: "{:,.0f}".format(x)))


In [ ]:
# Ecriture Silver en Parquet
tables_silver = {
    "clients":      clients_pd,
    "employes":     employes_pd,
    "fournisseurs": fournisseurs_pd,
    "produits":     produits_pd,
    "ventes":       ventes_clean,
}

for nom, df in tables_silver.items():
    path = SILVER + "/" + nom
    os.makedirs(path, exist_ok=True)
    df.to_parquet(path + "/" + nom + ".parquet", index=False)
    print("Ecrit : " + path + "/" + nom + ".parquet  (" + str(len(df)) + " lignes)")

print("")
print("Toutes les tables Silver sont enregistrees !")


In [ ]:
# Verification finale
print("=== VERIFICATION SILVER ===")
for nom in tables_silver.keys():
    df_check = pd.read_parquet(SILVER + "/" + nom + "/" + nom + ".parquet")
    print(nom.upper() + " — " + str(len(df_check)) + " lignes | colonnes : " + str(list(df_check.columns)))

print("")
print("Notebook 03 termine !")
